# POE - Silver Layer

This notebook cleans and validates the MIMIC-IV `poe` Bronze table before storing it in the Silver layer.

`POE` contains provider-entered clinical orders linked to hospital admissions, including the order type, timing, status and the provider responsible where available.

The Silver layer focuses on missing values, duplicate orders, provider relationships and text consistency while preserving the original order structure.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

poe = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.poe_raw"
)

print(f"Rows: {poe.count():,}")
print(f"Columns: {len(poe.columns)}")

check_columns = [
    "poe_id",
    "poe_seq",
    "subject_id",
    "hadm_id",
    "ordertime",
    "order_type",
    "order_subtype",
    "transaction_type",
    "order_provider_id",
    "order_status"
]

total = poe.count()

for column in check_columns:
    nulls = poe.filter(F.col(column).isNull()).count()

    if nulls > 0:
        pct = (nulls / total) * 100
        print(f"{column}: {nulls:,} nulls ({pct:.2f}%)")

### Missing Value Findings

- `order_subtype` is missing for many orders, which suggests it is only used for certain order types rather than being a required field.
- `order_provider_id` is missing for a small number of records, so some orders cannot be linked to a provider.
- These null values will be preserved rather than filled without a reliable source.

In [0]:
# Check duplicates and key relationships
duplicate_orders = (
    poe
    .groupBy("poe_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

provider = spark.table("workspace.silver.provider")

unmatched_providers = (
    poe
    .filter(F.col("order_provider_id").isNotNull())
    .select(F.col("order_provider_id").alias("provider_id"))
    .distinct()
    .join(provider.select("provider_id"), on="provider_id", how="left_anti")
    .count()
)

# Check self-referencing discontinued order IDs
poe_ids = poe.select("poe_id").distinct()

unmatched_discontinue_of = (
    poe
    .filter(F.col("discontinue_of_poe_id").isNotNull())
    .select(F.col("discontinue_of_poe_id").alias("poe_id"))
    .distinct()
    .join(poe_ids, on="poe_id", how="left_anti")
    .count()
)

unmatched_discontinued_by = (
    poe
    .filter(F.col("discontinued_by_poe_id").isNotNull())
    .select(F.col("discontinued_by_poe_id").alias("poe_id"))
    .distinct()
    .join(poe_ids, on="poe_id", how="left_anti")
    .count()
)

print(f"Duplicate poe_id values: {duplicate_orders:,}")
print(f"Unmatched provider IDs: {unmatched_providers:,}")
print(f"Unmatched discontinue_of_poe_id values: {unmatched_discontinue_of:,}")
print(f"Unmatched discontinued_by_poe_id values: {unmatched_discontinued_by:,}")

### Relationship Findings

- `poe_id` is unique with no duplicate orders.
- All populated provider IDs successfully match the `PROVIDER` table.
- All `discontinued_by_poe_id` references are valid.
- A small number of `discontinue_of_poe_id` values do not exist in the demo `POE` table. These will be reviewed but preserved, as they may refer to orders outside the demo subset.

In [0]:
# Inspect unmatched discontinued-order references
unmatched_discontinue_rows = (
    poe
    .filter(F.col("discontinue_of_poe_id").isNotNull())
    .join(
        poe_ids,
        poe["discontinue_of_poe_id"] == poe_ids["poe_id"],
        how="left_anti"
    )
    .select(
        "poe_id",
        "subject_id",
        "hadm_id",
        "ordertime",
        "discontinue_of_poe_id",
        "order_type",
        "order_status"
    )
)

display(unmatched_discontinue_rows)

### Cleaning Decision

The unmatched `discontinue_of_poe_id` values appear to be otherwise valid orders and may reference records that are not included in the demo subset.

These references will be preserved in Silver. A validation flag will be added so unmatched discontinued-order references remain visible without removing potentially valid clinical information.

In [0]:
# Standardise text fields
text_columns = [
    "order_type",
    "order_subtype",
    "transaction_type",
    "order_status"
]

poe_silver = poe

for column in text_columns:
    poe_silver = poe_silver.withColumn(
        column,
        F.when(F.trim(F.col(column)) == "", None)
         .otherwise(F.trim(F.col(column)))
    )

# Add discontinued-order reference validation
valid_poe_ids = (
    poe
    .select(F.col("poe_id").alias("_valid_poe_id"))
    .distinct()
)

poe_silver = (
    poe_silver
    .join(
        valid_poe_ids,
        poe_silver["discontinue_of_poe_id"] == valid_poe_ids["_valid_poe_id"],
        how="left"
    )
    .withColumn(
        "discontinue_reference_valid",
        F.when(F.col("discontinue_of_poe_id").isNull(), F.lit(None).cast("boolean"))
         .otherwise(F.col("_valid_poe_id").isNotNull())
    )
    .drop("_valid_poe_id")
)

print(f"POE Silver: {poe_silver.count():,} rows")
display(poe_silver.limit(10))

### Silver Transformation Check

The POE data has been lightly standardised and the discontinued-order relationship has been validated.

A quality flag has been added so unmatched discontinued-order references remain visible without removing otherwise valid records.

In [0]:
# Check reference validation results
(
    poe_silver
    .groupBy("discontinue_reference_valid")
    .count()
    .orderBy("discontinue_reference_valid")
    .show()
)

### Silver Validation Result

The discontinued-order validation behaves as expected.

Most POE records do not contain a discontinued-order reference, 10,486 references successfully match another POE record, and 24 remain unmatched in the demo dataset. These unmatched references are preserved and flagged rather than removed.

In [0]:
# Save POE Silver table
(
    poe_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.poe")
)

bronze_count = spark.table("workspace.bronze.poe_raw").count()
silver_count = spark.table("workspace.silver.poe").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")


### Silver Result

`POE` was successfully written to the Silver layer with all Bronze records preserved.

The table now includes light text standardisation and a validation flag for discontinued-order references, while keeping unmatched demo records for traceability.